In [ ]:
import numpy
import pandas
import time

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

import rateana

# bookkeeping of extracted databases
TRIGGER_DATABASE_FILES = {
  'Run1': None,
  'Run2': '/Users/triozzi/Analysis/icarus-trigger/data/dbs/TriggerDatabase-Runs09646-09796-5145B67C-20260714.pickle.gz',
  'Run3': '/Users/triozzi/Analysis/icarus-trigger/data/dbs/TriggerDatabase-Runs11879-12037-5145B67C-20260714.pickle.gz',
  'Run4': '/Users/triozzi/Analysis/icarus-trigger/data/dbs/TriggerDatabase-Runs13047-13205-5145B67C-20260714.pickle.gz',
  'Run5': '/Users/triozzi/Analysis/icarus-trigger/data/dbs/TriggerDatabase-Runs13864-13974-5145B67C-20260714.pickle.gz',
}

# dump all plots here, and gitignore it
PRINT_PATH = 'plots/'

## Trigger rates from database

Sourced from Gianluca Petrillo (petrillo@slac.stanford.edu) on Thu 13 Mar 2025.

#### Investigate a file

Some general settings to make the following plots more automated and consistent:

- `GateNames` lists the name of the gates we have data about;
- `SignalBeams` lists the name of the beams we have signal data about (`'BNB'`, `'NuMI'`).

Data is organized per run, with one column dedicated to each gate type.
This makes it easier to compare on-beam and off-beam rates.
Each run includes its start and end timestamps (first and last triggered event in the run), the run duration in seconds, and the number of recorded events.

In [ ]:
# high-level settings for each stream
BeamInfo = {
    'BNB':         dict(name='BNB',         beamName='BNB',         color=('darkorange', 1), sourceIndex=1, reference={ 'spill': 1.6, 'rate': 4.0,    'POT/spill': 3.5e12 }, veto=4.0, hasSignal=True,  scale=1.0, lw=1.5),
    'NuMI':        dict(name='NuMI',        beamName='NuMI',        color=('darkblue', 1), sourceIndex=2, reference={ 'spill': 9.5, 'rate': 0.9375, 'POT/spill': 4.0e13 }, veto=4.0, hasSignal=True,  scale=1.0, lw=1.5),
    'BNBoffbeam':  dict(name='BNBoffbeam',  beamName='BNB',         color=('orange', 1), sourceIndex=3, reference={ 'spill': 1.6, 'rate': 4.0,    'POT/spill': 3.5e12 }, veto=4.0, hasSignal=False, scale=1.0, lw=1.0),
    'NuMIoffbeam': dict(name='NuMIoffbeam', beamName='NuMI',        color=('blue', 1), sourceIndex=4, reference={ 'spill': 9.5, 'rate': 0.9375, 'POT/spill': 4.0e13 }, veto=4.0, hasSignal=False, scale=1.0, lw=1.0),
    'calibration': dict(name='calibration', beamName='calibration', color=('mediumpurple', 0.4), sourceIndex=5, reference={ 'spill': 1.6, 'rate': 1.00,   'POT/spill': 0.0  },   veto=4.0, hasSignal=False, scale=1.0, lw=1.0),
}

GateNames = [ 'BNB', 'BNBoffbeam', 'NuMI', 'NuMIoffbeam', 'calibration' ] # here for the order

# give a proper name to each stream
GateToName = [ None ] * (len(BeamInfo) + 1)
for beamInfo in BeamInfo.values(): GateToName[beamInfo['sourceIndex']] = beamInfo['name']

In [ ]:
# load dataset
TrigDB = pandas.read_pickle(TRIGGER_DATABASE_FILES['Run2'])
TrigDB.info()
RunList = sorted(TrigDB.run_number.unique())
print(f"Loaded {len(TrigDB)} triggers in {len(RunList)} runs"
      f" between {TrigDB.run_number.iloc[0]} ({time.ctime(TrigDB.beam_timestamp.iloc[0]/1e9)})"
      f" and {TrigDB.run_number.iloc[-1]} ({time.ctime(TrigDB.beam_timestamp.iloc[-1]/1e9)})."
)

In [ ]:
# load dataset
TrigDB = pandas.read_pickle(TRIGGER_DATABASE_FILES['Run2'])
TrigDB.info()
RunList = sorted(TrigDB.run_number.unique())
print(f"Loaded {len(TrigDB)} triggers in {len(RunList)} runs"
      f" between {TrigDB.run_number.iloc[0]} ({time.ctime(TrigDB.beam_timestamp.iloc[0]/1e9)})"
      f" and {TrigDB.run_number.iloc[-1]} ({time.ctime(TrigDB.beam_timestamp.iloc[-1]/1e9)})."
)

# what beam are there?
SignalBeams = [ beamName for beamName, beamInfo in BeamInfo.items() if beamInfo['hasSignal'] and (TrigDB.gate_type == beamInfo['sourceIndex']).any() ]

# give a proper name to each stream
gateToName = [ None ] * (len(BeamInfo) + 1)
for beamInfo in BeamInfo.values(): gateToName[beamInfo['sourceIndex']] = beamInfo['name']

beamToOnbeamGate = numpy.zeros(len(GateNames)+1).astype(int)
beamToOffbeamGate = numpy.zeros(len(GateNames)+1).astype(int)
for beamInfo in BeamInfo.values():
    (beamToOnbeamGate if beamInfo['hasSignal'] else beamToOffbeamGate)[BeamInfo[beamInfo['beamName']]['sourceIndex']] = beamInfo['sourceIndex']

for beamName in SignalBeams:
    beam_type = BeamInfo[beamName]['sourceIndex']
    print(f"{beamName} (#{beam_type}) has on-beam stream {gateToName[beamToOnbeamGate[beam_type]]} ({beamToOnbeamGate[beam_type]})"
          f" and off-beam stream {gateToName[beamToOffbeamGate[beam_type]]} ({beamToOffbeamGate[beam_type]}).")
    
# this array maps the gate type index to the beam (BNB or NuMI), expressed as the type of the on-beam gate:
GateToBeamMap = numpy.zeros(max(info['sourceIndex'] for info in BeamInfo.values()) + 1, dtype=int)
for info in BeamInfo.values(): GateToBeamMap[info['sourceIndex']] = BeamInfo[info['beamName']]['sourceIndex']

# discover the gate content of our data
AvailableGates = sorted(TrigDB.gate_type.unique().astype(int))
AvailableBeams = sorted(numpy.unique(GateToBeamMap[TrigDB.gate_type]).astype(int))
print(f"Data has {len(AvailableGates)} gates:", ", ".join(map(gateToName.__getitem__, AvailableGates)))
print(f"Data has {len(AvailableBeams)} beams:", ", ".join(map(gateToName.__getitem__, AvailableBeams)))

In [ ]:
RunPeriods = {
    'Run1':       dict(tag='Run1', name='ICARUS Run1', firstRun= 8460,  lastRun= 8554, prescale=dict(BNB=200, NuMI=60, BNBoffbeam=20, NuMIoffbeam=20), color='teal'       ),
    'Run2':       dict(tag='Run2', name='ICARUS Run2', firstRun= 9301,  lastRun=10097, prescale=dict(BNB=200, NuMI=60, BNBoffbeam=20, NuMIoffbeam=20), color='orange'     ),
    'Run3':       dict(tag='Run3', name='ICARUS Run3', firstRun=11806,  lastRun=12037, prescale=dict(BNB= 40, NuMI=30, BNBoffbeam=20, NuMIoffbeam=20), color='forestgreen'),
    'Run4':       dict(tag='Run4', name='ICARUS Run4', firstRun=12962,  lastRun=13272, prescale=dict(BNB= 40, NuMI=30, BNBoffbeam=20, NuMIoffbeam=20), color='purple'     ),
    'Run5':       dict(tag='Run5', name='ICARUS Run5', firstRun=13758,  lastRun=99999, prescale=dict(BNB= 40, NuMI=30, BNBoffbeam=20, NuMIoffbeam=20), color='navy'    ),
    # "preparation" included changes in trigger (adders), beam gate delays, minimum bias prescale
    'Run3prep':   dict(tag='Run3prep',   name='ICARUS Run3 preparation (to 2024-03-30)', firstRun=11806, lastRun=11834, prescale=dict(BNB=200, NuMI=60, BNBoffbeam=20, NuMIoffbeam=20)), # before the change of beam gate delays
    'Run3stable': dict(tag='Run3stable', name='ICARUS Run3 (2024-03-30 on)',             firstRun=11835, lastRun=12037, prescale=dict(BNB= 40, NuMI=30, BNBoffbeam=20, NuMIoffbeam=20)), # after the change of beam gate delays
}

In [ ]:
# prepare for the analysis
RunInfo = TrigDB.groupby('run_number').agg(
    startTimeStamp=pandas.NamedAgg('beam_timestamp', 'min'),
    endTimeStamp=pandas.NamedAgg('beam_timestamp', 'max'),
    events=pandas.NamedAgg('beam_timestamp', 'size'),
)
RunInfo['duration'] = (RunInfo.endTimeStamp - RunInfo.startTimeStamp) / 1e9

# add a column for each gate type (`<gate type>_events`)
# with the number of light-triggered events collected for that gate type
RunStats = (
    pandas.concat(
        [ RunInfo ] + [
            gateData.groupby('run_number').size().to_frame(gateToName[gate_type] + '_events') for gate_type, gateData in TrigDB[TrigDB.trigger_type == 0].groupby('gate_type')
        ],
        axis='columns'
    ).fillna(0).astype(int) # some runs don't have any event in a certain gate
    .reset_index() # `run_number` back to a normal column
)

# also add the excess of beam-induced events. Should be positive...
for beamName in SignalBeams:
    RunStats[beamName + '_excess_events'] = RunStats[beamName + '_events'] - RunStats[beamName + 'offbeam_events']

# now add average event rates
for streamName in [ gateToName[name] for name in AvailableGates ] + [ name + '_excess' for name in SignalBeams ]:
    try:
        RunStats[streamName + '_rate'] = (RunStats[streamName + '_events'] - 1) / RunStats.duration
        RunStats[streamName + '_rate_error'] = numpy.sqrt(RunStats[streamName + '_events']) / RunStats.duration
    except KeyError:
        print('cathced a `KeyError` in', streamName)
        print('there is no `_rate` here, and maybe this was intended (e.g., for `calibration`.)')

In [ ]:
runToStart = lambda run: RunStats.startTimeStamp.iloc[RunStats.run_number.searchsorted(run)]
for periodInfo in RunPeriods.values():
    print(f"{periodInfo['name']} started with run {periodInfo['firstRun']} on {time.ctime(runToStart(periodInfo['firstRun'])/1e9)}")

In [ ]:
fig, ax = plt.subplots(layout='constrained', figsize=(5, 3))
for gate_type in AvailableGates:
    gateName = gateToName[gate_type]
    if gateName != 'calibration':
        ax.scatter(
            RunStats.startTimeStamp / 1e9, RunStats[gateName + '_rate'] * 1000.0,
            label=f"{gateName} ({sum(RunStats[gateName + '_events'].astype(bool))} runs)",
            s=2, color=BeamInfo[gateName]['color'],
        )
for periodInfo in RunPeriods.values():
    if 'color' not in periodInfo: continue
    if periodInfo['firstRun'] > RunStats.iloc[-1].run_number: continue
    if periodInfo['lastRun'] <= RunStats.iloc[0].run_number: continue
    periodStart = runToStart(periodInfo['firstRun'])/1e9
    periodEnd = (runToStart(periodInfo['lastRun'] + 1) if (periodInfo['lastRun'] < RunStats.iloc[-1].run_number) else RunStats.iloc[-1].endTimeStamp)/1e9
    ax.axvspan(periodStart, periodEnd, color=( periodInfo['color'], 0.2 ), label=periodInfo['name'])
ax.set(
    title="Average trigger rate",
    xlabel="run start time  [ s ]",
    ylabel="average trigger rate  [ mHz ]",
)
ax.grid('x', color='lightgray', linestyle=':')
ax.legend()
# savefig("AverageTriggerRate", fig)

In [ ]:
CapRateAt = 0.3 # mHz
fig, ax = plt.subplots(layout='constrained', figsize=(5, 3))
for beam_type in AvailableBeams:
    beamName = gateToName[beam_type]
    if beamName != 'calibration':
        ax.scatter(
            RunStats.startTimeStamp / 1e9, 
            numpy.fmin(RunStats[beamName + '_excess_rate'], CapRateAt) * 1000.0,
            label=f"{beamName} ({sum(RunStats[beamName + '_excess_events'].astype(bool))} runs)",
            s=3, color=BeamInfo[beamName]['color'],
        )
for periodInfo in RunPeriods.values():
    if 'color' not in periodInfo: continue
    if periodInfo['firstRun'] > RunStats.iloc[-1].run_number: continue
    if periodInfo['lastRun'] <= RunStats.iloc[0].run_number: continue
    periodStart = runToStart(periodInfo['firstRun'])/1e9
    periodEnd = (runToStart(periodInfo['lastRun'] + 1) if (periodInfo['lastRun'] < RunStats.iloc[-1].run_number) else RunStats.iloc[-1].endTimeStamp)/1e9
    ax.axvspan(periodStart, periodEnd, color=( periodInfo['color'], 0.2 ), label=periodInfo['name'])
ax.set(
    title=f"Beam-induced trigger rate (capped at {CapRateAt*1000.0} mHz)",
    xlabel="run start time  [ s ]",
    ylabel="average trigger rate  [ mHz ]",
    ylim=( -10.0, CapRateAt * 1000.0), # mHz
)
ax.grid('x', color='lightgray', linestyle=':')
ax.legend()
# savefig("BeamInducedTriggerRate", fig)

In extracting the following statistics, we exclude runs that are too short (< 30').

In [ ]:
# for period, info in RunPeriods.items():
#     periodData = RunStats[RunStats.run_number.between(info['firstRun'], info['lastRun']) & (RunStats.duration >= 1800.0)]
#     if periodData.empty: continue
#     print(f"{info['name']:12} ({len(periodData):3} runs between {periodData.run_number.iloc[0]:5d} and {periodData.run_number.iloc[-1]:5d}) excess events:", end='')
#     for beam_type in AvailableBeams:
#         beamData = periodData[
#             (periodData[gateToName[beamToOnbeamGate[beam_type]] + '_events'] >= 100)
#             & (periodData[gateToName[beamToOffbeamGate[beam_type]] + '_events'] >= 100)
#         ]
#         if beamData.empty: continue
#         beamStats = dict(
#             runs=len(periodData), # qualifying runs
#             events=sum(periodData[gateToName[beamToOnbeamGate[beam_type]] + '_excess_events']),
#             duration=sum(periodData.duration),
#         )
#         print(f" {gateToName[beam_type]}: {beamStats['events']/beamStats['duration']*1000:4.1f} mHz ({beamStats['events']:6d} events from {beamStats['runs']:3d} runs)", end='')
#     # for beam
#     print('.')
# # for run period


For off-beam minimum bias events, we count all indistinctively.

In [ ]:
triggerToName = [ 'light-based', 'minimum bias' ]
gateColWidth = max(map(len, filter(None, gateToName)))
for period, info in RunPeriods.items():
    periodData = TrigDB[TrigDB.run_number.between(info['firstRun'], info['lastRun'])]
    if periodData.empty: continue
        
    triggerColWidth = max(max(map(len, triggerToName)), len(info['name']))
    print(f"{info['name']:<{triggerColWidth}s} ({len(periodData):3} runs between {periodData.run_number.iloc[0]:5d} and {periodData.run_number.iloc[-1]:5d})")

    print(f"   {info['name']:<{triggerColWidth}s}", end='')
    for gate_type in AvailableGates: print(f"  {gateToName[gate_type]:>{gateColWidth}s}", end='')
    print()
    triggerCounts = periodData.groupby([ 'gate_type', 'trigger_type' ])['beam_timestamp'].apply(len)
    for trigger_type, triggerName in enumerate(triggerToName):
        print(f"   {triggerName:<{triggerColWidth}s}", end='')
        for gate_type in AvailableGates:
            try:             nEvents = int(triggerCounts.loc[gate_type, trigger_type])
            except KeyError: nEvents = 0
            print(f"  {nEvents:{gateColWidth}d}", end='')
        print()
    # for trigger type
# for run period


In [ ]:
RunPeriod = RunPeriods['Run2']
RunTag = RunPeriod['tag']

In [ ]:
BeamBinnings = {
    'BNB': {  'range': ( 0.064*-0.5, 0.064*40.5 ), 'binWidth': 0.064, },
    'NuMI': { 'range': ( 0.256*-0.5, 0.256*49.5 ), 'binWidth': 0.064, },
}

fig, axes = plt.subplots(len(SignalBeams), layout='constrained', squeeze=False, figsize=(4, 3.5 * len(SignalBeams)))
itAxis = iter(axes)
for iBeam, beamName in enumerate(SignalBeams):
    beamInfo = BeamInfo[beamName]
    binningInfo = BeamBinnings[beamName]

    periodData = TrigDB[(TrigDB.trigger_type == 0) & TrigDB.run_number.between(RunPeriod['firstRun'], RunPeriod['lastRun'])]

    data = periodData[periodData.gate_type == beamInfo['sourceIndex']]
    if len(data) == 0: continue
    ax = next(itAxis)[0]
    
    bins = numpy.arange(binningInfo['range'][0], binningInfo['range'][1], binningInfo['binWidth']) + 0.001
    offbeamName = beamName + 'offbeam'
    
    plotData = data.triggerFromBeamGate/1e3 - beamInfo['veto']
    artists = ax.hist(
        plotData, bins=bins,
        histtype="step",
        color=beamInfo['color'],
        label=f"{beamInfo['name']} ({len(plotData)} events)",
        )[2]
    artists[0].set(linewidth=3.0)

    try:
        offbeamInfo = BeamInfo[beamName + 'offbeam']
    except KeyError: pass
    else:
        bkgrData = periodData[periodData.gate_type == offbeamInfo['sourceIndex']]
        plotBkgrData = bkgrData.triggerFromBeamGate/1e3 - offbeamInfo['veto']
        artists = ax.hist(
            plotBkgrData, bins=bins,
            histtype="step",
            color=offbeamInfo['color'],
            label=f"off-beam ({len(plotBkgrData)} events)",
            )[2]
        artists[0].set(linewidth=3.0)
    # off-beam
    
    ax.set(
        xlabel="trigger time - beam gate time  [ µs ]",
        ylabel=f"events  [ / {binningInfo['binWidth']*1_000:g} ns ]",
    )
    # ax.axvline(0.5)
    
    ax.grid(color='lightgray', ls=':')
    ax.legend()
    print(f"{beamInfo['name']} range: {min(plotData)*1000} to {max(plotData)*1000} ns")
fig.suptitle(f"{RunPeriod['name']}: trigger time relative to beam gate")
# savefig("TriggerTimeInBeamGate", fig)

In [ ]:
for RunPeriod in RunPeriods.values():
    RunTag = RunPeriod['tag']
    BeamBinnings = {
        'BNB': {  'range': ( 0.064*-0.5, 0.064*40.5 ), 'binWidth': 0.032, },
        'NuMI': { 'range': ( 0.256*-0.5, 0.256*46.5 ), 'binWidth': 0.064, },
    }
    
    for iBeam, beamName in enumerate(SignalBeams):
        beamInfo = BeamInfo[beamName]
        binningInfo = BeamBinnings[beamName]
    
        periodData = TrigDB[(TrigDB.trigger_type == 0) & TrigDB.run_number.between(RunPeriod['firstRun'], RunPeriod['lastRun'])]
    
        data = periodData[periodData.gate_type == beamInfo['sourceIndex']]
        if len(data) == 0: continue
        fig, ax = plt.subplots(layout='constrained', figsize=(3, 3))
        
        bins = numpy.arange(binningInfo['range'][0], binningInfo['range'][1], binningInfo['binWidth']) + 0.001
        offbeamName = beamName + 'offbeam'
        
        plotData = data.triggerFromBeamGate/1e3 - beamInfo['veto']
        artists = ax.hist(
            plotData, bins=bins,
            histtype="step",
            color=beamInfo['color'],
            label=f"{beamInfo['name']} ({len(plotData)} events)",
            )[2]
        artists[0].set(linewidth=3.0)
    
        try:
            offbeamInfo = BeamInfo[beamName + 'offbeam']
        except KeyError: pass
        else:
            bkgrData = periodData[periodData.gate_type == offbeamInfo['sourceIndex']]
            plotBkgrData = bkgrData.triggerFromBeamGate/1e3 - offbeamInfo['veto']
            artists = ax.hist(
                plotBkgrData, bins=bins,
                histtype="step",
                color=offbeamInfo['color'],
                label=f"off-beam ({len(plotBkgrData)} events)",
                )[2]
            artists[0].set(linewidth=3.0)
        # off-beam
        
        ax.set(
            title=f"{RunPeriod['name']} {beamInfo['name']}: trigger time relative to beam gate",
            xlabel="trigger time - beam gate time  [ µs ]",
            ylabel=f"events  [ / {binningInfo['binWidth']*1_000:g} ns ]",
        )
        # ax.axvline(0.5)
        
        ax.grid(color='lightgray', ls=':')
        ax.legend()
        print(f"{beamInfo['name']} range: {min(plotData)*1000:g} to {max(plotData)*1000:g} ns")
        # savefig(f"TriggerTimeInBeamGate-{beamInfo['name']}", fig)
    # for beam
# for period

#### Trigger rates per physics run

In [ ]:
fig, axes = plt.subplots(layout='constrained', ncols=2, figsize=(4*2, 3), sharey=True)

### show run2 rate extraction first
ax = axes[0]

# run2 
db = rateana.process.LoadTriggerDatabase(TRIGGER_DATABASE_FILES['Run2'])
df = rateana.process.ProcessTriggerDatabase(db)
df_bnb = rateana.process.FilterTriggerDataframe(df, beam='BNB')
df_numi = rateana.process.FilterTriggerDataframe(df, beam='NuMI')

BNB_halfdur  = pandas.to_timedelta(df_bnb.duration  / 2.0, unit='s')
NuMI_halfdur = pandas.to_timedelta(df_numi.duration / 2.0, unit='s')

BNB_xmid  = df_bnb.startTimeStamp_hr  + BNB_halfdur
NuMI_xmid = df_numi.startTimeStamp_hr + NuMI_halfdur

commonStyle = dict(ls='', capsize=0)

ax.errorbar(BNB_xmid, df_bnb['BNB_rate'] * 1000.0,
    yerr=df_bnb['BNB_rate_error'] * 1000.0, xerr=BNB_halfdur,
    c='C0', marker='o', ms=4, label='BNB on-beam', **commonStyle)
ax.errorbar(BNB_xmid, df_bnb['BNBoffbeam_rate'] * 1000.0,
    yerr=df_bnb['BNBoffbeam_rate_error'] * 1000.0, xerr=BNB_halfdur,
    c='C0', marker='o', ms=4, markerfacecolor='none', label='BNB off-beam', **commonStyle)

ax.errorbar(NuMI_xmid, df_numi['NuMI_rate'] * 1000.0,
    yerr=df_numi['NuMI_rate_error'] * 1000.0, xerr=NuMI_halfdur,
    c='C1', marker='s', ms=3.75, label='NuMI on-beam', **commonStyle)
ax.errorbar(NuMI_xmid, df_numi['NuMIoffbeam_rate'] * 1000.0,
    yerr=df_numi['NuMIoffbeam_rate_error'] * 1000.0, xerr=NuMI_halfdur,
    c='C1', marker='s', ms=3.75, markerfacecolor='none', label='NuMI off-beam', **commonStyle)

ax.set(
  ylim = (0, 400),
  xlim = (df_bnb.startTimeStamp_hr.min(), df_bnb.startTimeStamp_hr.max()),
  ylabel = 'avg trigger rate\nper run [mHz]'
)
leg = ax.legend(fontsize=10, ncol=2, title='Run2', columnspacing=1, loc='upper center'); leg.get_title().set_fontsize(12)
# ax.set_xticks(ax.get_xticks())
# ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=12)
ax.xaxis.set_major_locator(matplotlib.dates.AutoDateLocator(minticks=5, maxticks=3))
plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=12)

### run-by-run
ax = axes[1]

runs = []
BNB_rates, BNB_offbeam_rates = [], []
NuMI_rates, NuMI_offbeam_rates = [], []
MinBias_rates = []

for run, path in TRIGGER_DATABASE_FILES.items():
    if path is not None:
        db = rateana.process.LoadTriggerDatabase(path)
        df = rateana.process.ProcessTriggerDatabase(db)

        runs.append(run)

        # BNB
        df_bnb = rateana.process.FilterTriggerDataframe(df, beam='BNB')
        BNB_rates.append(numpy.median(df_bnb['BNB_rate'] * 1000.0))
        BNB_offbeam_rates.append(numpy.median(df_bnb['BNBoffbeam_rate'] * 1000.0))
        minbias_all = numpy.median(df_bnb['BNB_minbias_rate'] * 1000.0) + numpy.median(df_bnb['BNBoffbeam_minbias_rate'] * 1000.0)
        
        # NuMI
        if run in ['Run1', 'Run2', 'Run3']:
            df_numi = rateana.process.FilterTriggerDataframe(df, beam='NuMI')
            NuMI_rates.append(numpy.median(df_numi['NuMI_rate'] * 1000.0))
            NuMI_offbeam_rates.append(numpy.median(df_numi['NuMIoffbeam_rate'] * 1000.0))
            minbias_all += numpy.median(df_numi['NuMI_minbias_rate'] * 1000.0) + numpy.median(df_numi['NuMIoffbeam_minbias_rate'] * 1000.0)
        else:
            NuMI_rates.append(0)
            NuMI_offbeam_rates.append(0)

        MinBias_rates.append(minbias_all)

runs = numpy.array(runs)
BNB_rates = numpy.array(BNB_rates)
BNB_offbeam_rates = numpy.array(BNB_offbeam_rates)
NuMI_rates = numpy.array(NuMI_rates)
NuMI_offbeam_rates = numpy.array(NuMI_offbeam_rates)

RUN1_BNB = 93+21
RUN1_BNB_OFFBEAM = 92
RUN1_NuMI = 70+49
RUN1_NuMI_OFFBEAM = 72
RUN1_MINBIAS = 269
# ax.plot(range(len(runs)), BNB_rates, marker='o', ms=4, c='C0', lw=0.75)
# ax.plot(range(len(runs)), BNB_offbeam_rates, marker='o', ms=4, c='C0', markerfacecolor='none', lw=0.75, ls='--')
# ax.plot(range(len(runs[NuMI_rates > 10])), NuMI_rates[NuMI_rates > 10], marker='s', ms=3.75, c='C1', lw=0.75)
# ax.plot(range(len(runs[NuMI_offbeam_rates > 10])), NuMI_offbeam_rates[NuMI_offbeam_rates > 10], marker='s', ms=3.75, c='C1', markerfacecolor='none', lw=0.75, ls='--')
# ax.plot(range(len(runs)), MinBias_rates, lw=1.25, c='black', ls='-', label='min-bias')

runs = numpy.insert(runs, 0, 'Run1')
BNB_rates = numpy.insert(BNB_rates, 0, RUN1_BNB)
BNB_offbeam_rates = numpy.insert(BNB_offbeam_rates, 0, RUN1_BNB_OFFBEAM)
NuMI_rates = numpy.insert(NuMI_rates, 0, RUN1_NuMI)
NuMI_offbeam_rates = numpy.insert(NuMI_offbeam_rates, 0, RUN1_NuMI_OFFBEAM)
MinBias_rates = numpy.insert(MinBias_rates, 0, RUN1_MINBIAS)

ax.plot(range(len(runs)), BNB_rates, marker='o', ms=4, c='C0', lw=0.75)
ax.plot(range(len(runs)), BNB_offbeam_rates, marker='o', ms=4, c='C0', markerfacecolor='none', lw=0.75, ls='--')
ax.plot(range(len(runs[NuMI_rates > 10])), NuMI_rates[NuMI_rates > 10], marker='s', ms=3.75, c='C1', lw=0.75)
ax.plot(range(len(runs[NuMI_offbeam_rates > 10])), NuMI_offbeam_rates[NuMI_offbeam_rates > 10], marker='s', ms=3.75, c='C1', markerfacecolor='none', lw=0.75, ls='--')
ax.plot(range(len(runs)), MinBias_rates, lw=1.25, c='black', ls='-', label='min-bias')

# gfx
ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xticklabels(['Run1', 'Run2', 'Run3', 'Run4', 'Run5'])
ax.legend(loc='lower right', fontsize=10)

# ax2 = ax.twinx()
# ax2.plot(range(len(runs)), BNB_rates+BNB_offbeam_rates+NuMI_rates+NuMI_offbeam_rates, lw=1., c='black', label='total')
# ax2.set(
#     ylim = (0, 500)
# )
# ax2.legend()

PLOT_NAME = 'trigger_rates'
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.pdf', dpi=300)
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.png', dpi=300)

In [ ]:
BNB_rates,BNB_offbeam_rates,NuMI_rates,NuMI_offbeam_rates,MinBias_rates

In [ ]:
db = rateana.process.LoadTriggerDatabase(TRIGGER_DATABASE_FILES['Run2'])
df = rateana.process.ProcessTriggerDatabase(db)
df_bnb = rateana.process.FilterTriggerDataframe(df, beam='BNB')
df_numi = rateana.process.FilterTriggerDataframe(df, beam='NuMI')

(df_bnb.startTimeStamp_hr.min(), df_bnb.startTimeStamp_hr.max())

In [ ]:
BNB_rates+BNB_offbeam_rates+NuMI_rates+NuMI_offbeam_rates+MinBias_rates

In [ ]:
BNB_rates+BNB_offbeam_rates+NuMI_rates+NuMI_offbeam_rates

In [ ]:
BNB_rates-BNB_offbeam_rates, NuMI_rates-NuMI_offbeam_rates

#### Beam profiles per run

In [ ]:
import matplotlib.ticker as mticker

def SetSciYAxis(ax):
    fmt = mticker.ScalarFormatter(useMathText=True)
    fmt.set_scientific(True)
    fmt.set_powerlimits((0, 0))  # force sci notation regardless of magnitude
    ax.yaxis.set_major_formatter(fmt)

fig = plt.figure(layout='constrained', figsize=(3.5*2, 2.5*2))
gs = fig.add_gridspec(nrows=4, ncols=2, height_ratios=[0.08, 1, 0.08, 1])

for titleRow, plotRow, period in [(0, 1, 'Run2'), (2, 3, 'Run3')]:
    # centered row title, spanning both columns
    titleAx = fig.add_subplot(gs[titleRow, :])
    titleAx.axis('off')
    titleAx.text(0.5, 0, period, ha='center', va='bottom', fontsize=17)

    db = rateana.process.LoadTriggerDatabase(TRIGGER_DATABASE_FILES[period])

    # BNB
    ax = fig.add_subplot(gs[plotRow, 0])
    ax = rateana.hat.PlotTriggerHatPlot(ax, db, 'BNB', period)
    SetSciYAxis(ax)
    ax.set_ylabel('triggers /\n64 ns')
    ax.legend(fontsize=9.5)

    # NuMI
    ax = fig.add_subplot(gs[plotRow, 1])
    ax = rateana.hat.PlotTriggerHatPlot(ax, db, 'NuMI', period)
    SetSciYAxis(ax)
    ax.legend(fontsize=9.5)

ax.set(xlabel='trigger time - beam gate time  [µs]')

PLOT_NAME = 'trigger_hat_plots_Run23'
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.pdf', dpi=300)
fig.savefig(f'{PRINT_PATH}{PLOT_NAME}.png', dpi=300)